# Week 10: A Command-Line RAG Assistant

Same logic as `rag_assistant.py`. Requires `LLM_API_KEY` and `LLM_MODEL` in a `.env` file (see `.env.example`) — this calls a real LLM to generate the grounded answer. Retrieval reuses the persistent collection Week 9 built (`data/processed/chroma`) — run `examples/week-09/build_passage_index.py` first if you haven't already.

`answer_question()` itself is provider-agnostic; only `call_llm` below is Anthropic-specific.

In [ ]:
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

from ai_finance_course.rag import answer_question
from ai_finance_course.vector_store import get_or_create_collection

PERSIST_PATH = Path("data/processed/chroma")
COLLECTION_NAME = "sample_passages"
ANTHROPIC_MESSAGES_URL = "https://api.anthropic.com/v1/messages"


def call_llm(prompt: str) -> str:
    """The one Anthropic-specific piece; answer_question() itself is provider-agnostic."""
    with httpx.Client(timeout=60.0) as client:
        response = client.post(
            ANTHROPIC_MESSAGES_URL,
            headers={
                "x-api-key": os.environ["LLM_API_KEY"],
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": os.environ["LLM_MODEL"],
                "max_tokens": 1024,
                "messages": [{"role": "user", "content": prompt}],
            },
        )
        response.raise_for_status()
        data = response.json()
        for block in data["content"]:
            if block["type"] == "text":
                return block["text"]
        raise ValueError(f"No text block in response: {data}")


load_dotenv()
collection = get_or_create_collection(PERSIST_PATH, COLLECTION_NAME)

## Ask a Question

In [ ]:
question = "did the company beat earnings expectations?"

result, evidence = answer_question(question, collection, call_llm, n_results=3)
print(f"Q: {question}")
print(f"A: {result.answer}")

## Show the Sources

In [ ]:
if result.citations:
    for citation in result.citations:
        chunk = evidence[citation - 1]
        print(f"[{citation}] ({chunk['metadata']['ticker']}, {chunk['metadata']['doc_type']}) {chunk['text']}")
else:
    print("No sources cited.")

## Ask a Filtered Question

In [ ]:
filtered_result, filtered_evidence = answer_question(
    question, collection, call_llm, n_results=3, where={"ticker": "AAPL"}
)
print(f"A (AAPL only): {filtered_result.answer}")